# Wish Generation LLM Evaluation

This notebook evaluates different LLM models for personalized wish generation.

## Metrics Tracked
- **latency_ms**: Time per prompt
- **tokens_in / tokens_out**: Inference cost basis
- **quality_score**: LLM-as-a-judge focusing on:
  - Warmth and personalization
  - Age-appropriateness
  - Safety (no hallucinated dangerous/offensive content)
- **cost_1m_requests**: Cost for 1M similar requests

In [ ]:
from santa.eval import (
    calculate_cost_1m_requests,
    calculate_self_hosted_cost,
    evaluate_quality_with_judge,
    generate_wish,
    generate_wish_quality_judge_prompt,
    get_available_models,
    get_judge_client,
    get_test_profiles,
    get_test_recommendations_for_wish,
    load_env_from_repo_root,
    setup_mlflow,
)

load_env_from_repo_root()


In [ ]:
setup_mlflow("wish_generation_eval")

In [ ]:
test_profiles = get_test_profiles()
test_recommendations = get_test_recommendations_for_wish()

test_recommendations

In [ ]:
models_to_evaluate = get_available_models(
    include_openai=True,
    include_token_factory=True,
    include_self_hosted=True,
)

In [ ]:
judge_client = get_judge_client()

In [ ]:
import re
from pathlib import Path

import mlflow
import pandas as pd

test_profiles = get_test_profiles()
test_recommendations = get_test_recommendations_for_wish()

results = []
agg_results = []

MLFLOW_AVAILABLE = bool(mlflow.get_tracking_uri())

# Evaluate all configured models
for model_config in models_to_evaluate:
    model_name = model_config.name
    client = model_config.client
    print(f"Evaluating model: {model_name}")

    with mlflow.start_run(run_name=f"{model_name}"):
        per_calls = []

        for kid_profile, gift_recommendation in zip(
            test_profiles, test_recommendations, strict=True
        ):
            try:
                wish, metrics = generate_wish(client, kid_profile, gift_recommendation)

                quality_score = None
                quality_rationale = None
                if judge_client:
                    judge_prompt = generate_wish_quality_judge_prompt(kid_profile, wish.text)
                    quality_score, quality_rationale = evaluate_quality_with_judge(
                        judge_client, judge_prompt
                    )

                tokens_in = metrics.get("tokens_in", 0) or 0
                tokens_out = metrics.get("tokens_out", 0) or 0
                latency_ms = metrics.get("latency_ms", 0) or 0

                # Calculate costs: dynamic for self-hosted, fixed for provider models
                if model_config.is_self_hosted and model_config.cost_infra_per_hour:
                    cost_per_1m_in, cost_per_1m_out = calculate_self_hosted_cost(
                        latency_ms=latency_ms,
                        tokens_in=tokens_in,
                        tokens_out=tokens_out,
                        cost_infra_per_hour=model_config.cost_infra_per_hour,
                    )
                else:
                    cost_per_1m_in = model_config.cost_per_1m_tokens_in
                    cost_per_1m_out = model_config.cost_per_1m_tokens_out

                cost_in_1m = round(cost_per_1m_in, 1)
                cost_out_1m = round(cost_per_1m_out, 1)

                cost_1m_requests = calculate_cost_1m_requests(
                    cost_per_1m_in=cost_per_1m_in,
                    cost_per_1m_out=cost_per_1m_out,
                    tokens_in=tokens_in,
                    tokens_out=tokens_out,
                )

                record = {
                    "model": model_name,
                    "kid_id": kid_profile.id,
                    "latency_ms": round(latency_ms, 0),
                    "tokens_in": tokens_in,
                    "tokens_out": tokens_out,
                    "quality_score": quality_score,
                    "cost_in_1m": cost_in_1m,
                    "cost_out_1m": cost_out_1m,
                    "cost_1m_requests": cost_1m_requests,
                }
                per_calls.append(record)
                results.append(record)

                if MLFLOW_AVAILABLE:
                    with mlflow.start_run(run_name=f"{kid_profile.id}", nested=True):
                        mlflow.log_metric("latency_ms", record["latency_ms"])
                        mlflow.log_metric("tokens_in", tokens_in)
                        mlflow.log_metric("tokens_out", tokens_out)
                        if quality_score is not None:
                            mlflow.log_metric("quality_score", quality_score)
                        mlflow.log_metric("cost_in_1m", cost_in_1m)
                        mlflow.log_metric("cost_out_1m", cost_out_1m)
                        mlflow.log_metric("cost_1m_requests", cost_1m_requests)
                        mlflow.set_tags(
                            {
                                "kid_id": kid_profile.id,
                                "task": "wish_generation",
                            }
                        )
                        mlflow.log_dict(
                            {
                                "wish": wish.text,
                                "quality_score": quality_score,
                                "quality_rationale": quality_rationale,
                            },
                            "wish_report.json",
                        )

            except Exception as e:
                err_rec = {
                    "model": model_name,
                    "kid_id": kid_profile.id,
                    "error": str(e),
                }
                per_calls.append(err_rec)
                results.append(err_rec)

        df_model = pd.DataFrame([r for r in per_calls if "error" not in r])
        if not df_model.empty:
            agg = {
                "model": model_name,
                "latency_ms": round(df_model["latency_ms"].mean(), 0),
                "tokens_in": df_model["tokens_in"].mean(),
                "tokens_out": df_model["tokens_out"].mean(),
                "quality_score": df_model["quality_score"].mean(),
                "cost_in_1m": round(df_model["cost_in_1m"].mean(), 1),
                "cost_out_1m": round(df_model["cost_out_1m"].mean(), 1),
                "cost_1m_requests": round(df_model["cost_1m_requests"].mean(), 1),
                "calls": len(df_model),
            }
            agg_results.append(agg)

            out_dir = Path("data/evaluation")
            out_dir.mkdir(parents=True, exist_ok=True)
            model_name_path = re.sub(r"[^\w\-]", "-", model_name)
            out_path = out_dir / f"03_wish_eval-{model_name_path}.csv"
            df_model.to_csv(out_path, index=False)
            print(f"Saved per-call results for {model_name} -> {out_path}")

            if MLFLOW_AVAILABLE:
                mlflow.log_metric("latency_ms", agg["latency_ms"])
                mlflow.log_metric("tokens_in", agg["tokens_in"])
                mlflow.log_metric("tokens_out", agg["tokens_out"])
                mlflow.log_metric("quality_score", agg["quality_score"])
                mlflow.log_metric("cost_in_1m", agg["cost_in_1m"])
                mlflow.log_metric("cost_out_1m", agg["cost_out_1m"])
                mlflow.log_metric("cost_1m_requests", agg["cost_1m_requests"])
                mlflow.log_metric("calls", agg["calls"])
                mlflow.set_tags({"task": "wish_generation"})
        else:
            print(f"No successful calls for model {model_name}")

In [ ]:
# Final aggregated summary
df_results = pd.DataFrame(results)
df_agg = pd.DataFrame(agg_results)
print("Per-model aggregate summary:")
print(df_agg.round(3))